# Notebook 05 — Light & Temperature Extremes Processing

This notebook extracts, cleans, and standardizes parameters representing light availability (`Solar_Radiation`), maximum temperature (`Temperature_Max`), minimum temperature (`Temperature_Min`), and diurnal temperature range (`Temp_Range`).

## 1. Import Packages & Local Configuration

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Ensure project root is in the path
sys.path.append(str(Path(os.getcwd()).parent))

import config
print("Processed Data Folder:", config.PROCESSED_DIR)

Processed Data Folder: D:\A1\coco\Coconut_Yield_Prediction\processed


## 2. Load Raw Weather and Climate Data

In [2]:
raw_weather_path = config.RAW_DIR / "weather_raw.csv"
if not raw_weather_path.exists():
    raise FileNotFoundError(f"{raw_weather_path.name} not found. Run Notebook 02 first.")

df_raw = pd.read_csv(raw_weather_path)
print("Loaded raw weather data shape:", df_raw.shape)

Loaded raw weather data shape: (728, 11)


## 3. Clean and Transform Light & Extremes

We calculate the Diurnal Temperature Range (DTR):
$$\text{Temp\_Range} = \text{Temperature\_Max} - \text{Temperature\_Min}$$
DTR is a major factor in plant transpiration and stress.

In [3]:
df_light_temp = df_raw[["District", "Year", "Month", "Solar_Radiation", "Temperature_Max", "Temperature_Min"]].copy()

# Calculate range
df_light_temp["Temp_Range"] = df_light_temp["Temperature_Max"] - df_light_temp["Temperature_Min"]

# Physical limits check (temperature range must be positive, solar rad must be non-negative)
df_light_temp.loc[df_light_temp["Solar_Radiation"] < 0, "Solar_Radiation"] = np.nan
df_light_temp.loc[df_light_temp["Temp_Range"] < 0, "Temp_Range"] = np.nan

# Impute missing if any by grouping by district/month
if df_light_temp.isnull().sum().sum() > 0:
    medians = df_light_temp.groupby(["District", "Month"])[["Solar_Radiation", "Temperature_Max", "Temperature_Min", "Temp_Range"]].transform("median")
    df_light_temp = df_light_temp.fillna(medians)

print(df_light_temp.describe())

              Year      Month  Solar_Radiation  Temperature_Max  \
count   728.000000  728.00000       728.000000       728.000000   
mean   2022.000000    7.00000        19.295536        31.547280   
std       2.001375    3.74423         2.368843         2.006092   
min    2019.000000    1.00000        13.220000        28.430000   
25%    2020.000000    4.00000        17.950000        30.110000   
50%    2022.000000    7.00000        19.230000        30.850000   
75%    2024.000000   10.00000        21.000000        32.390000   
max    2025.000000   13.00000        24.850000        37.440000   

       Temperature_Min  Temp_Range  
count       728.000000  728.000000  
mean         23.178022    8.369258  
std           2.115320    3.054368  
min          19.250000    3.980000  
25%          21.200000    5.962500  
50%          23.900000    7.410000  
75%          24.960000   10.730000  
max          26.710000   17.580000  


## 4. Export Cleaned Dataset

In [4]:
out_path = config.PROCESSED_DIR / "light_temp_clean.csv"
df_light_temp.to_csv(out_path, index=False)
print(f"[SUCCESS] Cleaned light & temperature extremes dataset saved to {out_path.name}")

[SUCCESS] Cleaned light & temperature extremes dataset saved to light_temp_clean.csv
